The [GPT from scratch](/aiml-common/lectures/nlp/transformers/gpt-from-scratch/gpt-from-scratch) section builds a decoder-only transformer, the architecture GPT uses, in code.

Compared to the RNNs of the earlier chapters, the transformer architecture makes two changes.

1. It removes all recurrent connections. During training, all positions of a sequence are then processed in parallel, and at inference the model reads the whole prompt in one parallel pass. Generating text is still sequential, because each new token depends on the tokens before it. Since the processing no longer follows the order of the tokens, the order has to be encoded explicitly, through positional information, either in the input embeddings or inside attention. The [cross-attention and position encoding](/aiml-common/lectures/nlp/transformers/cross-attention-position-encoding/index) section covers the methods.

2. It uses attention to _adjust_ context-free embeddings, such as the word2vec embeddings, based on the context of the input sequence. Attention can sit in the encoder, in the decoder, or in both. The name comes from what the mechanism does: each token attends to the other tokens of the input sequence, and the attention weights decide how much each token contributes to the update.

The figure below shows the two transformer architectures you meet in this chapter. The original transformer has an encoder and a decoder [@Vaswani2017-vd]. The encoder reads the whole source sentence. The decoder generates the output one token at a time and reads the encoder output through cross-attention. GPT and most current language models keep only the decoder.

Both architectures stack $N$ identical layers, and each layer alternates two kinds of computation. In the attention sublayers, shown in orange, tokens exchange information, and these are the only places where they do. In the MLP sublayers, shown in blue, each token is transformed on its own, with the same weights for every token. A residual connection adds the input of each sublayer to its output.

![Encoder-decoder and decoder-only transformer architectures. Each layer contains attention sublayers, where tokens exchange information, and an MLP sublayer that transforms each token on its own; residual connections add each sublayer's input to its output](images/transformer-architectures.svg)

*Encoder-decoder and decoder-only transformers. Layer normalization is not drawn.*

Most language models released since 2020 are decoder-only transformers that refine the right half of this figure. The [model releases](/research/models/index) page lists the major models with their technical reports. The reports below draw the architecture of their model, and you can compare each drawing with the figure above.

- GPT-1, Figure 1 (left): a 12-layer decoder-only transformer with masked self-attention, trained to predict the next token [@Radford_undated-cp].
- Llama 3, Figure 1: a dense decoder-only transformer, close to the original design. Its attention uses grouped-query attention, where several query heads share one key-value head [@Grattafiori2024-ew].
- Mixtral, Figure 1: the MLP of each layer becomes a mixture of experts. A router sends each token to 2 of 8 MLPs and adds their weighted outputs [@Jiang2024-ik].
- DeepSeek-V3, Figure 2: a mixture of experts in place of the MLP, and multi-head latent attention, which compresses the keys and values so that less memory is needed during generation [@DeepSeek-AI2024-fj].

In all four, the split of the figure above still holds: attention is where tokens exchange information, and the MLP or its mixture of experts acts on each token on its own.

To see what self-attention changes, consider the token _bears_ in three sentences:

* I love _bears_
* She _bears_ the pain
* _Bears_ won the game

The token has a different meaning in each sentence: an animal, tolerating something, and a sports team. Assume a tokenizer that maps all three occurrences to the same token ID (for example, an uncased tokenizer). Their lookup embeddings are then identical, although position embeddings may differ. A bidirectional encoder can produce different contextual representations by combining information from the surrounding tokens. A causal decoder can use only the prefix through the current token; in particular, the first token of the third sentence cannot yet use "won the game".

## Simple self-attention mechanism

The simplest way to let the other tokens affect a context-free representation is a dot product between the input token $x_i$ and every token of the context. This is called simple _self-attention_. The word "self" means that each token attends to the tokens of its own sequence. The simple version uses the input vectors directly, with no learned transformation.

The next figure shows the input context matrix $X$ with dimensions $T \times d$, where $T$ is the length of the input context and $d$ is the embedding dimension. In this example each token occupies a row with $d$ features with $d=4$, and there are $T=3$ tokens. Real models are much larger: [Qwen3.8-27B](https://huggingface.co/Qwen/Qwen3.8-27B) uses $d=5120$ and a context of up to $T=262{,}144$ tokens.

![Simple self-attention: X times X transpose gives the T by T scores S, a softmax over each row gives the weights A, and A times X gives the output, one updated row per token](images/simple-self-attention.svg)

*Simple self-attention on a context of three tokens. The input $X$ plays all three roles: it is compared with itself to give the scores, and it is also what the weights average.*

Inputs usually also carry a batch dimension $B$, so $X$ is a tensor of shape $B \times T \times d$. This section ignores batching.

Self-attention updates the embedding $x_i$ of the $i$-th token in three steps.

The first step computes the attention scores $s_{ij} = x_i^T x_j$. Each score is the dot product of the input token with one token $j$ of the context, including the input token itself. With 3 tokens there are 3 scores per token, so the scores form a $3 \times 3$ matrix.

The second step passes each row of scores through a softmax to obtain the _attention weights_ $\alpha_{ij}$. The softmax maps a vector to a vector of values between 0 and 1 that sum to 1. It is applied across the tokens of the context, so each input token gets $T=3$ attention weights. Each row is a categorical distribution over the available positions, but it is not a prediction of a class label or a calibrated explanation. For finite, unmasked scores it makes all attention weights positive and makes the tokens of the context compete for weight:

$$
\alpha_{ij} \ge 0, \qquad \sum_{j=1}^T \alpha_{ij} = 1.
$$

A large attention weight on one token comes at the expense of the weights on the other tokens. In the limiting case, when token $i$ puts all of its weight on itself ($\alpha_{ii} = 1$ and $\alpha_{ij} = 0$ for $j \ne i$), its embedding stays unchanged.

The third step uses the attention weights to form a weighted sum of the token embeddings. The result is the new embedding of the input token, which now includes information from the other embeddings of the input sequence:

$$
\hat x_i = \sum_{j=1}^T \alpha_{ij} x_j.
$$

All three steps fit into one equation, with the softmax applied to each row:

$$
\hat X = \mathtt{softmax}(XX^T)X.
$$

The code below runs the three steps on a random context with $T=3$ and $d=4$.

In [ ]:
import torch

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False)

T, d = 3, 4
X = torch.randn(T, d)  # input context, one token per row

S = X @ X.T  # attention scores s_ij
A = torch.softmax(S, dim=-1)  # attention weights, softmax over each row
X_hat = A @ X  # output embeddings

print("scores S:\n", S)
print("weights A:\n", A)
print("row sums:", A.sum(dim=-1))
print("output shape:", tuple(X_hat.shape))

This version has no trainable parameters, and it has a weakness. The score of a token with itself, $x_i^T x_i = \lVert x_i \rVert^2$, is usually the largest in its row for the independent random embeddings used here, so each token attends mostly to itself. In the output above, the first token puts almost all of its weight on itself. The third token is the exception: its own score (1.07) is smaller than its score with the second token (1.26), because its embedding is short.

The exception becomes rare as the embedding dimension grows. For random embeddings with unit-variance entries, $\lVert x_i \rVert^2$ is about $d$, while the dot product of two different tokens has a typical size of only $\sqrt d$. The next cell counts, over 1000 random contexts of $T=8$ tokens, how often a token's largest weight is on itself.

In [ ]:
for d in (4, 16, 64):
    X = torch.randn(1000, 8, d)  # 1000 random contexts of 8 tokens
    A = torch.softmax(X @ X.transpose(1, 2), dim=-1)
    self_is_max = (A.argmax(dim=-1) == torch.arange(8)).float().mean()
    self_weight = A.diagonal(dim1=1, dim2=2).mean()
    print(f"d={d:3d}: largest weight on itself {self_is_max:.0%}, mean self weight {self_weight:.2f}")

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(1)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, d in zip(axes, (4, 64)):
    X = torch.randn(8, d)
    A = torch.softmax(X @ X.T, dim=-1)
    im = ax.imshow(A, cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"attention weights, d = {d}")
    ax.set_xlabel("token j")
    ax.set_ylabel("token i")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

For these random embeddings, the weight matrix at $d=64$ is close to the identity, so $\hat X \approx X$. This is not a universal property of real embeddings. Learned projections allow the model to replace the fixed similarity function with a trainable compatibility function.

## The output as a move inside the convex hull

The weights of each row are non-negative and sum to one, so $x_i = \sum_j \alpha_{ij} x_i$. Subtracting this from the output gives

$$
\hat x_i = x_i + \Delta_i, \qquad \Delta_i = \sum_{j=1}^T \alpha_{ij}\,(x_j - x_i).
$$

You can read the update as a move. The token starts at $x_i$ and is displaced by $\Delta_i$, the weighted average of the vectors that point from $x_i$ to each token of the context. The move is bounded. A weighted average with non-negative weights that sum to one always lies in the _convex hull_ of the tokens, the smallest convex region that contains all of them. Simple self-attention can pull a token toward the tokens it attends to, but it cannot push the token away from another token or carry it outside that region.

The next figure shows the move of $x_1$ in a context of four tokens with $d=2$.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

X = torch.tensor([[0.1, 1.1], [1.2, 0.9], [1.1, -0.7], [-0.9, -0.1]])
a = torch.softmax(X @ X[0], dim=-1)  # weights of token 1
x_hat = a @ X
x1 = X[0]

fig, ax = plt.subplots(figsize=(6.4, 5.2))
ax.add_patch(Polygon(X.numpy(), closed=True, fc="#dbe8f7", ec="#4f86c6", lw=1.5,
                     label="convex hull of the tokens"))
for j in range(1, len(X)):
    ax.annotate("", X[j].tolist(), x1.tolist(),
                arrowprops=dict(arrowstyle="->", color="#9ca3af", lw=1, ls="--"))
    mid = x1 + 0.55 * (X[j] - x1)
    ax.text(mid[0], mid[1], rf"$\alpha_{{1{j + 1}}}={a[j]:.2f}$", fontsize=10, color="#4b5563",
            ha="center", va="bottom", bbox=dict(fc="white", ec="none", pad=1))
ax.annotate("", x_hat.tolist(), x1.tolist(),
            arrowprops=dict(arrowstyle="-|>", color="#d9822b", lw=3, mutation_scale=18))
mid = x1 + 0.5 * (x_hat - x1)
ax.text(mid[0] - 0.05, mid[1] + 0.07, r"$\Delta_1$", color="#d9822b", fontsize=14)
ax.scatter(X[:, 0], X[:, 1], s=70, c="#3d556e", zorder=3)
for k, p in enumerate(X):
    ax.text(p[0] + 0.07, p[1] + 0.08, rf"$x_{k + 1}$", fontsize=14)
ax.text(x1[0] - 0.2, x1[1] + 0.2, rf"$\alpha_{{11}}={a[0]:.2f}$", fontsize=10, color="#4b5563")
ax.scatter([x_hat[0]], [x_hat[1]], s=90, c="#d9822b", marker="D", zorder=4)
ax.text(x_hat[0] + 0.06, x_hat[1] - 0.17, r"$\hat x_1$", fontsize=14, color="#d9822b")
ax.set_aspect("equal")
ax.set_xlim(-1.3, 1.8)
ax.set_ylim(-1.0, 1.55)
ax.grid(alpha=0.2)
ax.legend(loc="lower right", fontsize=9)
plt.show()


*The weights are the softmax of the scores $x_1^T x_j$. Token $x_1$ keeps 0.44 of its weight on itself and gives 0.39 to $x_2$, so $\hat x_1$ moves only part of the way toward $x_2$ and stays inside the hull.*

A full transformer layer adds the attention output to $x_i$ through a residual connection, so there the update is a literal addition. In that layer the tokens pass through learned linear maps before they are averaged, so the update is no longer confined to the convex hull of the input tokens.

Think of attention as a soft lookup in a collection of records. Each record has a key used for matching and a value used for retrieval. A query is compared with every available key, and the retrieved values are combined using the resulting weights. In self-attention each token supplies all three: a query, a key and a value. The [next section](/aiml-common/lectures/nlp/transformers/singlehead-self-attention/index) develops this idea.

## PyTorch reference

| PyTorch class | Description |
|---|---|
| [`nn.Embedding`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Embedding.html) | A lookup table of embeddings for a fixed vocabulary. |
| [`nn.Linear`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html) | Affine map of the input: $y = xA^T + b$. |
| [`nn.Softmax`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Softmax.html) | Softmax over one dimension of a tensor. |
| [`nn.MultiheadAttention`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.MultiheadAttention.html) | Attention with several heads in parallel. |
| [`nn.TransformerEncoderLayer`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.TransformerEncoderLayer.html) | Self-attention followed by a feed-forward network. |
| [`nn.TransformerDecoderLayer`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.TransformerDecoderLayer.html) | Self-attention, cross-attention and a feed-forward network. |

Key references: [@Bahdanau2014-lc; @Luong2015-dd; @Vaswani2017-vd]